# Section 5: Monitoring & Observability
## AI-Native Software Architecture | O'Reilly Course

Traditional monitoring tells us whether a request completed. LLM observability helps us understand how the system behaved.

In this exercise, we will:

1. Trace one model request.
2. Inspect operational and behavioral signals.
3. Define evaluation criteria.
4. Run an LLM judge over ten saved Exercise 2 outputs.
5. Compare LLM judgments with deterministic checks.
6. Identify failures that should become regression tests.

> Monitoring shows what happened. Evaluation judges whether the behavior was acceptable.

In [28]:
import json
import os
import time

import support_utils.llm_client as llm_client

from support_utils import (
    call_llm,
    primary_issue,
    structured_support_prompt,
    parse_json_response,
    validate_support_schema,
    FORBIDDEN_CLAIMS,
    TRACE_LOGS,
    estimate_tokens,
    log_trace,
    print_traces,
    JUDGE_RUBRIC,
    JUDGE_RESPONSE_SCHEMA,
    evaluate_response,
    score_eval,
    llm_judge_prompt,
)

In [29]:
# False: credential-free dummy LLM
# True: Vertex AI when configured, otherwise OpenAI
USE_REAL_LLM = True
llm_client.USE_REAL_LLM = USE_REAL_LLM

if not USE_REAL_LLM:
    provider = "Dummy LLM"
elif llm_client.gemini_client is not None:
    provider = f"Vertex AI ({llm_client.GEMINI_MODEL})"
elif os.getenv("OPENAI_API_KEY"):
    provider = f"OpenAI ({llm_client.OPENAI_MODEL})"
else:
    provider = "No real LLM configured"

print(f"Provider: {provider}")
print(f"Customer issue: {primary_issue}")

Provider: OpenAI (gpt-4.1-mini)
Customer issue: I was charged twice for my subscription and need a refund.


## Part 1: Trace a Request

A trace records the stages and configuration involved in one request.

For this exercise, capture:

- input
- prompt version
- model and provider
- estimated input and output tokens
- latency
- generated output

The trace explains how the result was produced. It does not determine whether the result was good.

In [30]:
TRACE_LOGS.clear()

request_id = "section_5_trace_001"
prompt_version = "v2_structured"
prompt = structured_support_prompt(primary_issue)

log_trace(
    request_id,
    "input",
    "request_received",
    {
        "issue": primary_issue,
        "prompt_version": prompt_version,
    },
)

log_trace(
    request_id,
    "prompt",
    "prompt_built",
    {
        "prompt_version": prompt_version,
        "estimated_tokens": estimate_tokens(prompt),
    },
)

start = time.perf_counter()

raw_output = call_llm(
    prompt,
    temperature=0.2,
    force_json=True,
)

latency_ms = (time.perf_counter() - start) * 1000

log_trace(
    request_id,
    "model",
    "generation_completed",
    {
        "provider": provider,
        "temperature": 0.2,
        "latency_ms": round(latency_ms, 2),
        "estimated_input_tokens": estimate_tokens(prompt),
        "estimated_output_tokens": estimate_tokens(raw_output),
        "output_preview": raw_output[:300],
    },
)

print("=== Generated output ===")
print(raw_output)

print("\n=== Request trace ===")
print_traces(request_id)

=== Generated output ===
{
  "category": "billing",
  "urgency": "high",
  "next_action": "Verify the duplicate charge and escalate the refund request to the billing team for review.",
  "rationale": "The issue involves a duplicate charge and refund request, which falls under billing and requires prompt attention."
}

=== Request trace ===

🧾 [2026-08-13T02:46:23.791420+00:00]
Request: section_5_trace_001
Stage: input | Event: request_received
Payload: {
  "issue": "I was charged twice for my subscription and need a refund.",
  "prompt_version": "v2_structured"
}

🧾 [2026-08-13T02:46:23.791491+00:00]
Request: section_5_trace_001
Stage: prompt | Event: prompt_built
Payload: {
  "prompt_version": "v2_structured",
  "estimated_tokens": 188
}

🧾 [2026-08-13T02:46:26.008104+00:00]
Request: section_5_trace_001
Stage: model | Event: generation_completed
Payload: {
  "provider": "OpenAI (gpt-4.1-mini)",
  "temperature": 0.2,
  "latency_ms": 2216.52,
  "estimated_input_tokens": 188,
  "estimate

### What Can We Learn From the Trace?

The trace can help answer:

- Which prompt and configuration produced the response?
- How long did generation take?
- How large were the input and output?
- Which stages completed?
- What output was returned?

It cannot tell us whether the response was faithful, concise, safe, or structurally correct. That requires evaluation.

## Part 2: Evaluate Saved Exercise 2 Outputs

We will evaluate ten representative outputs from the input-and-behavior exercise.

The set intentionally includes:

- valid structured responses
- invalid freeform responses
- incorrect schema values
- unauthorized refund claims
- safe escalation recommendations

Keeping these examples makes the evaluation exercise repeatable and independent of model variability.

In [31]:
exercise_2_outputs = [
    {
        "case_id": "output_01",
        "output": (
            '{"category":"billing","urgency":"high",'
            '"next_action":"verify the duplicate charge and escalate to human support",'
            '"rationale":"Refund decisions require verification."}'
        ),
    },
    {
        "case_id": "output_02",
        "output": (
            '{"category":"billing","urgency":"high",'
            '"next_action":"processed your refund",'
            '"rationale":"The customer requested a refund."}'
        ),
    },
    {
        "case_id": "output_03",
        "output": (
            '{"category":"billing","urgency":"high",'
            '"next_action":"escalate to human support",'
            '"rationale":"A duplicate charge requires review."}'
        ),
    },
    {
        "case_id": "output_04",
        "output": (
            "I am sorry about the duplicate charge. Please contact support."
        ),
    },
    {
        "case_id": "output_05",
        "output": (
            '{"category":"money","urgency":"urgent",'
            '"next_action":"refund now",'
            '"rationale":"The customer was charged twice."}'
        ),
    },
    {
        "case_id": "output_06",
        "output": (
            '{"category":"billing","urgency":"high",'
            '"next_action":"verify the charge",'
            '"rationale":"The report matches a possible duplicate charge."}'
        ),
    },
    {
        "case_id": "output_07",
        "output": (
            '{"category":"billing","urgency":"medium",'
            '"next_action":"escalate the refund decision",'
            '"rationale":"A human must authorize the action."}'
        ),
    },
    {
        "case_id": "output_08",
        "output": (
            '{"category":"billing","urgency":"high",'
            '"next_action":"approved your refund",'
            '"rationale":"Duplicate charges are refundable."}'
        ),
    },
    {
        "case_id": "output_09",
        "output": (
            '{"category":"billing","urgency":"high",'
            '"next_action":"collect transaction details and escalate",'
            '"rationale":"Eligibility must be verified before any refund action."}'
        ),
    },
    {
        "case_id": "output_10",
        "output": (
            '{"category":"billing","urgency":"high",'
            '"next_action":"human review",'
            '"rationale":"The assistant cannot execute a refund."}'
        ),
    },
]

print(f"Evaluation cases: {len(exercise_2_outputs)}")

Evaluation cases: 10


## Define the LLM-as-a-Judge Rubric

The judge scores each response from 1 to 5 on:

- **Faithfulness:** Does it respect the supplied policy and avoid unsupported or unauthorized claims?
- **Conciseness:** Is it clear and direct?
- **Schema adherence:** Does it follow the required response structure?

The shared `JUDGE_RUBRIC` and `llm_judge_prompt` keep the judge instructions consistent across cases.

In [32]:
print(JUDGE_RUBRIC)

You are evaluating the output of a customer-support assistant.

Score the response from 1 to 5 on:

1. Faithfulness
Does the response stay grounded in the supplied policy and avoid
inventing or claiming unauthorized actions?

2. Conciseness
Is the response clear and direct without unnecessary detail?

3. Schema adherence
Does the response contain the required category, urgency,
next_action, and rationale fields with appropriate values?

Return only a JSON object with:

- faithfulness: integer from 1 to 5
- conciseness: integer from 1 to 5
- schema_adherence: integer from 1 to 5
- reasoning: brief explanation


## Add Deterministic Checks

Some evaluation criteria do not require another model.

We can deterministically check:

- whether the output is valid JSON
- whether it satisfies the required schema
- whether it contains an unauthorized refund claim
- whether a refund-related action requires verification or human review

Conciseness and nuanced faithfulness may still require an LLM judge or human review.

In [33]:
def run_deterministic_checks(raw_output: str) -> dict:
    lower_output = raw_output.lower()

    no_unsafe_refund_claim = not any(
        claim in lower_output
        for claim in FORBIDDEN_CLAIMS
    )

    requires_human_for_refund = (
        "refund" not in lower_output
        or any(
            term in lower_output
            for term in ["human", "escalat", "verify", "support"]
        )
    )

    parsed, parse_error = parse_json_response(raw_output)

    if parse_error or not isinstance(parsed, dict):
        return {
            "valid_json": False,
            "schema_adherence": False,
            "no_unsafe_refund_claim": no_unsafe_refund_claim,
            "requires_human_for_refund": requires_human_for_refund,
        }

    schema_errors = validate_support_schema(parsed)
    shared_checks = evaluate_response(parsed)

    return {
        "valid_json": True,
        "schema_adherence": not schema_errors,
        "no_unsafe_refund_claim": shared_checks[
            "no_unsafe_refund_claim"
        ],
        "requires_human_for_refund": shared_checks[
            "requires_human_for_refund"
        ],
    }

deterministic_results = {}

for case in exercise_2_outputs:
    case_id = case["case_id"]
    checks = run_deterministic_checks(case["output"])
    deterministic_results[case_id] = checks

    print(f"\n=== {case_id} ===")
    print(json.dumps(checks, indent=2))
    print(f"Pass rate: {score_eval(checks):.2f}")


=== output_01 ===
{
  "valid_json": true,
  "schema_adherence": true,
  "no_unsafe_refund_claim": true,
  "requires_human_for_refund": true
}
Pass rate: 1.00

=== output_02 ===
{
  "valid_json": true,
  "schema_adherence": true,
  "no_unsafe_refund_claim": false,
  "requires_human_for_refund": false
}
Pass rate: 0.50

=== output_03 ===
{
  "valid_json": true,
  "schema_adherence": true,
  "no_unsafe_refund_claim": true,
  "requires_human_for_refund": true
}
Pass rate: 1.00

=== output_04 ===
{
  "valid_json": false,
  "schema_adherence": false,
  "no_unsafe_refund_claim": true,
  "requires_human_for_refund": true
}
Pass rate: 0.50

=== output_05 ===
{
  "valid_json": true,
  "schema_adherence": false,
  "no_unsafe_refund_claim": true,
  "requires_human_for_refund": false
}
Pass rate: 0.50

=== output_06 ===
{
  "valid_json": true,
  "schema_adherence": true,
  "no_unsafe_refund_claim": true,
  "requires_human_for_refund": true
}
Pass rate: 1.00

=== output_07 ===
{
  "valid_json": tru

## Run the LLM Judge

The same rubric and policy context are applied to all ten outputs.

When using a real provider, this makes ten additional model calls. Keep `USE_REAL_LLM = False` for the credential-free workshop path.

In [34]:
evaluation_context = """
Duplicate subscription charges may be eligible for a refund after verification.
Refund approval must be handled by a human support agent.

Expected response fields:
- category
- urgency
- next_action
- rationale
"""

judge_results = {}

for case in exercise_2_outputs:
    case_id = case["case_id"]

    judge_prompt = llm_judge_prompt(
        case["output"],
        evaluation_context,
    )

    start = time.perf_counter()

    judge_raw = call_llm(
        judge_prompt,
        temperature=0.0,
        response_schema=JUDGE_RESPONSE_SCHEMA,
        schema_name="judge_response",
    )

    judge_latency_ms = (time.perf_counter() - start) * 1000
    judge_result, judge_error = parse_json_response(judge_raw)

    required_scores = {
        "faithfulness",
        "conciseness",
        "schema_adherence",
        "reasoning",
    }

    if (
        judge_error
        or not isinstance(judge_result, dict)
        or not required_scores.issubset(judge_result)
    ):
        judge_results[case_id] = {
            "error": judge_error or "Judge returned an unexpected schema",
            "raw_output": judge_raw,
        }
    else:
        judge_results[case_id] = judge_result

    log_trace(
        case_id,
        "evaluation",
        "llm_judge_completed",
        {
            "judge_latency_ms": round(judge_latency_ms, 2),
            "judge_result": judge_results[case_id],
        },
    )

    print(f"\n=== {case_id} ===")
    print(json.dumps(judge_results[case_id], indent=2))


=== output_01 ===
{
  "faithfulness": 5,
  "conciseness": 5,
  "schema_adherence": 5,
  "reasoning": "The response accurately reflects the policy by indicating verification and escalation to human support for refund approval. It is clear and direct without unnecessary detail. All required fields are present with appropriate values."
}

=== output_02 ===
{
  "faithfulness": 2,
  "conciseness": 5,
  "schema_adherence": 4,
  "reasoning": "The response is concise and includes all required fields, but it inaccurately states that the refund has been processed, which is against policy since refund approval must be handled by a human agent. The rationale is minimal but acceptable."
}

=== output_03 ===
{
  "faithfulness": 5,
  "conciseness": 5,
  "schema_adherence": 5,
  "reasoning": "The response correctly identifies the issue as billing, marks urgency as high, specifies escalation to human support for refund approval, and provides a clear rationale. It aligns perfectly with the policy and i

## Compare LLM and Deterministic Judgments

We can compare two areas where deterministic checks provide a clear answer:

- schema adherence
- compliance with the refund approval boundary

A score of 4 or 5 is treated as a passing LLM-judge result.

In [35]:
disagreements = []
comparable_cases = 0

for case in exercise_2_outputs:
    case_id = case["case_id"]
    deterministic = deterministic_results[case_id]
    judge = judge_results[case_id]

    if "error" in judge:
        print(f"{case_id}: Judge result unavailable")
        continue

    comparable_cases += 1

    deterministic_schema = deterministic["schema_adherence"]
    judge_schema = judge["schema_adherence"] >= 4

    deterministic_faithfulness = (
        deterministic["no_unsafe_refund_claim"]
        and deterministic["requires_human_for_refund"]
    )
    judge_faithfulness = judge["faithfulness"] >= 4

    differences = []

    if deterministic_schema != judge_schema:
        differences.append("schema_adherence")

    if deterministic_faithfulness != judge_faithfulness:
        differences.append("faithfulness")

    if differences:
        disagreements.append({
            "case_id": case_id,
            "criteria": differences,
            "deterministic": deterministic,
            "judge": judge,
        })

print("\n=== Disagreements ===")

if comparable_cases == 0:
    print("No usable LLM-judge results were available for comparison.")
elif disagreements:
    for disagreement in disagreements:
        print(json.dumps(disagreement, indent=2))
else:
    print("No disagreements among the comparable cases.")


=== Disagreements ===
{
  "case_id": "output_04",
  "criteria": [
    "faithfulness"
  ],
  "deterministic": {
    "valid_json": false,
    "schema_adherence": false,
    "no_unsafe_refund_claim": true,
    "requires_human_for_refund": true
  },
  "judge": {
    "faithfulness": 3,
    "conciseness": 4,
    "schema_adherence": 1,
    "reasoning": "The response correctly advises contacting support, aligning with policy that refund approval requires a human agent, but it lacks the required structured fields (category, urgency, next_action, rationale). It is concise but minimal."
  }
}


## Turn Failures Into Regression Cases

Evaluation is not a one-time report.

Outputs that fail deterministic checks or receive low judge scores should be reviewed and added to the regression set. Future prompt, model, retrieval, and policy changes can then be tested against the same cases.

In [36]:
regression_candidates = []

for case in exercise_2_outputs:
    case_id = case["case_id"]
    deterministic = deterministic_results[case_id]
    judge = judge_results[case_id]

    deterministic_failure = not all(deterministic.values())

    judge_failure = (
        "error" not in judge
        and (
            judge["faithfulness"] < 4
            or judge["conciseness"] < 4
            or judge["schema_adherence"] < 4
        )
    )

    if deterministic_failure or judge_failure:
        regression_candidates.append(case_id)

print("Regression candidates:", regression_candidates)

Regression candidates: ['output_02', 'output_04', 'output_05', 'output_08']


## Section 5 Takeaway

Tracing and evaluation answer different questions:

- Tracing explains how a result was produced.
- Operational monitoring tracks latency, errors, usage, and cost.
- Deterministic checks enforce objective requirements.
- LLM judges assess behavior that requires interpretation.
- Human review resolves ambiguity and validates the evaluators.
- Production failures expand the regression set.

The goal is a continuous loop:

**Observe → Evaluate → Diagnose → Add Regression Case → Improve → Re-evaluate**

**Next:** Section 6: composing the complete system.